# 🚀 Crypto Analyzer v2 – KI-Kryptoanalyse in Google Colab

[![GitHub](https://img.shields.io/badge/GitHub-bademeischta%2Fcrypto__analyzer-blue)](https://github.com/bademeischta/crypto_analyzer)

> **Läuft komplett in Google Colab** – keine lokale Installation nötig!

## Was erwartet dich?

| Feature | Beschreibung |
|---|---|
| 📊 **Interaktive Charts** | Candlestick + EMA + Bollinger Bands + Volumen |
| 🤖 **KI-Vorhersage** | LightGBM mit Walk-Forward-Validation (kein Lookahead-Bias) |
| 🔄 **Backtesting** | Wie hätten die KI-Signale historisch performt? |
| 🔍 **Screener** | Mehrere Coins auf einmal scannen |
| 💭 **Sentiment** | Fear & Greed Index + Reddit-Stimmungsanalyse |

## Schnellstart (3 Schritte)

1. ▶️ **Zelle 1 ausführen** – Installiert alle Pakete (~1 Min)
2. ▶️ **Zelle 2 ausführen** – Lädt das Projekt von GitHub
3. ▶️ **Zelle 3 ausführen** – Startet die App und zeigt die URL

---
⚠️ **Disclaimer:** Diese App dient ausschließlich Bildungszwecken. Keine Finanzberatung.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ZELLE 1: Installation (einmalig ausführen)
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys

print('📦 Installiere Abhängigkeiten...')
print('   (Das dauert beim ersten Mal ~1-2 Minuten)')

packages = [
    'streamlit>=1.30.0,<2.0.0',
    'pyngrok>=7.0.0',
    'lightgbm>=4.0.0',
    'plotly>=5.18.0',
    'scikit-learn>=1.3.0',
    'joblib>=1.3.0',
    'PyYAML>=6.0',
    'requests>=2.31.0',
    'aiohttp>=3.9.0',
    'rich>=13.7.0',
    'pandas==2.0.3',
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    print('⚠️  Einige Pakete konnten nicht installiert werden:')
    print(result.stderr[-500:] if result.stderr else '(keine Details)')
else:
    print('✅ Alle Pakete installiert!')

# Versionsinfo
import importlib
checks = [('streamlit', 'streamlit'), ('lightgbm', 'lightgbm'),
          ('pandas', 'pandas'), ('plotly', 'plotly')]
print('\n📋 Installierte Versionen:')
for name, mod in checks:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', '?')
        print(f'   {name}: {ver}')
    except ImportError:
        print(f'   {name}: FEHLT ❌')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ZELLE 2: Projekt von GitHub laden
# ═══════════════════════════════════════════════════════════════════════
import os, sys, subprocess

REPO_URL   = 'https://github.com/bademeischta/crypto_analyzer'
REPO_BRANCH = 'main'  # oder 'claude/loving-lamport-np6oo7' für Entwicklungsversion
PROJECT_DIR = '/content/crypto_analyzer'

if os.path.exists(os.path.join(PROJECT_DIR, 'main.py')):
    print(f'✅ Projekt bereits vorhanden in {PROJECT_DIR}')
    print('   (Zum Aktualisieren: !git -C {PROJECT_DIR} pull)')
else:
    print(f'📥 Lade Projekt von GitHub...')
    result = subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        # Branch existiert eventuell nicht, versuche ohne Branch
        result = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, PROJECT_DIR],
            capture_output=True, text=True
        )
    if result.returncode != 0:
        print(f'❌ Fehler beim Klonen:\n{result.stderr}')
        raise RuntimeError('Git clone fehlgeschlagen!')
    print('✅ Projekt geladen!')

# Python-Pfad aktualisieren
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)

print(f'\n📁 Arbeitsverzeichnis: {os.getcwd()}')
print(f'🐍 Python: {sys.version.split()[0]}')

# Projektstruktur anzeigen
import pathlib
print('\n📂 Projektstruktur:')
for p in sorted(pathlib.Path(PROJECT_DIR).glob('*.py')) + sorted(pathlib.Path(PROJECT_DIR).glob('*.yaml')):
    print(f'   {p.name}')
print('   src/')
for p in sorted(pathlib.Path(PROJECT_DIR).glob('src/**/*.py')):
    print(f'   {str(p.relative_to(PROJECT_DIR))}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ZELLE 3: App starten (Streamlit + ngrok-Tunnel)
# ═══════════════════════════════════════════════════════════════════════
#
# OPTIONAL: ngrok-Auth-Token für stabilere Verbindung
# Kostenlosen Account erstellen: https://dashboard.ngrok.com/signup
# Token unter https://dashboard.ngrok.com/get-started/your-authtoken
#
NGROK_AUTH_TOKEN = ''  # Hier deinen Token eintragen (optional)

# ─────────────────────────────────────────────────────────────────────
import subprocess, sys, time, os, threading
from pyngrok import ngrok, conf

PROJECT_DIR = '/content/crypto_analyzer'
PORT = 8501

# ngrok konfigurieren
if NGROK_AUTH_TOKEN.strip():
    conf.get_default().auth_token = NGROK_AUTH_TOKEN.strip()
    print('✅ ngrok Auth-Token gesetzt')
else:
    print('ℹ️  Kein ngrok-Token → Anonym-Modus (funktioniert, aber begrenzt)')

# Alte Prozesse beenden falls vorhanden
try:
    ngrok.kill()
except Exception:
    pass

# Streamlit-Prozess starten
print('\n🚀 Starte Streamlit...')
streamlit_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', 'main.py',
        '--server.port', str(PORT),
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
        '--browser.gatherUsageStats', 'false',
        '--server.maxUploadSize', '50',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd=PROJECT_DIR,
)

# Auf Start warten
print('   Warte auf Streamlit-Start', end='', flush=True)
for _ in range(15):
    time.sleep(1)
    print('.', end='', flush=True)
    if streamlit_proc.poll() is not None:
        break
print()

if streamlit_proc.poll() is not None:
    print('❌ Streamlit konnte nicht starten!')
    output = streamlit_proc.stdout.read()
    print('Fehler-Output:', output.decode()[:1000] if output else '(leer)')
    raise RuntimeError('Streamlit-Start fehlgeschlagen')

print('✅ Streamlit läuft!')

# ngrok-Tunnel erstellen
print('\n🌐 Erstelle Tunnel...')
try:
    tunnel = ngrok.connect(PORT, 'http')
    public_url = tunnel.public_url

    print('\n' + '═'*60)
    print('  ✅ DEINE APP IST BEREIT!')
    print()
    print(f'  👉  {public_url}')
    print()
    print('═'*60)
    print('  Klicke den Link oben um die App zu öffnen.')
    print(f'  Lokaler Port: {PORT}')
    print()
    print('  Hinweise:')
    print('  • Die URL bleibt aktiv solange diese Colab-Sitzung läuft')
    print('  • Beim ersten Coin-Scan trainiert das KI-Modell (~20-60s)')
    print('  • Ergebnisse werden gecacht – spätere Scans sind schneller')

except Exception as e:
    print(f'⚠️  ngrok-Tunnel fehlgeschlagen: {e}')
    print('\nAlternative: Nutze die direkte Analyse in Zelle 5 unten.')

# Streamlit-URL als Variable speichern
try:
    app_url = public_url
except NameError:
    app_url = f'http://localhost:{PORT}'

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ZELLE 4: App-Status überwachen (optional)
# ═══════════════════════════════════════════════════════════════════════
# Zeigt den Status alle 60 Sekunden. Drücke ■ (Stop) zum Beenden.
# Die App läuft auch ohne diese Überwachungszelle weiter.

import time
from pyngrok import ngrok

print('📊 App-Überwachung aktiv. Drücke ■ zum Stoppen.\n')

try:
    while True:
        if streamlit_proc.poll() is not None:
            print('⚠️  Streamlit-Prozess wurde beendet!')
            break

        tunnels = ngrok.get_tunnels()
        status = f'✅ OK' if tunnels else '⚠️ Kein Tunnel'
        url = tunnels[0].public_url if tunnels else '–'

        print(f'[{time.strftime("%H:%M:%S")}] {status} | URL: {url}')
        time.sleep(60)

except KeyboardInterrupt:
    print('\nÜberwachung gestoppt. App läuft weiter.')

---
## Alternative: Direkte Analyse in Colab (ohne Streamlit-UI)

Falls der Tunnel nicht funktioniert, kannst du die Analyse auch direkt im Notebook durchführen.
Die Ergebnisse werden als interaktive Plotly-Charts angezeigt.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ZELLE 5: Direkte Analyse (ohne Streamlit)
# ═══════════════════════════════════════════════════════════════════════
import sys, os
PROJECT_DIR = '/content/crypto_analyzer'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# Plotly für Colab konfigurieren
import plotly.io as pio
pio.renderers.default = 'colab'

from pathlib import Path
from src.analysis.analyzer import CryptoAnalyzer

# ─── Hier Symbol ändern ──────────────────────────────────────────────
SYMBOL       = 'BTC'   # z.B. 'ETH', 'SOL', 'DOGE'
INTERVAL     = '1d'    # '1h', '4h', '1d'
LOOKBACK_DAYS = 365
# ─────────────────────────────────────────────────────────────────────

config_path = Path(PROJECT_DIR) / 'config.yaml'
analyzer = CryptoAnalyzer(config_path)

print(f'🔍 Analysiere {SYMBOL} ({INTERVAL}, {LOOKBACK_DAYS} Tage)...')
print('   (Erster Lauf trainiert das KI-Modell ~20–60s)\n')

result = analyzer.analyze(SYMBOL, INTERVAL, LOOKBACK_DAYS)

if result.error:
    print(f'❌ Fehler: {result.error}')
else:
    coin_name = result.market_data.get('name', SYMBOL)
    print(f'✅ Analyse für {coin_name} ({SYMBOL}) abgeschlossen!')
    print(f'   Datenpunkte: {len(result.ohlcv)}')
    if result.training_time_seconds > 1:
        print(f'   KI trainiert in: {result.training_time_seconds:.1f}s')

    print()

    # ── KI-Signal
    if result.prediction:
        p = result.prediction
        print('═'*50)
        print(f'  KI-Signal für {SYMBOL} (nächste {p.horizon_days} Tage):')
        print()
        if p.show_signal:
            print(f'  {p.direction_emoji}  {p.direction_label}')
            if p.confidence:
                print(f'  Konfidenz: {p.confidence:.1%}')
            print(f'  Volatilität: {p.volatility_label}')
        else:
            print(f'  Kein klares Signal ({p.no_signal_reason})')
        print('═'*50)
        print()

    # ── Marktdaten
    md = result.market_data
    if md:
        price = md.get('current_price')
        cap = md.get('market_cap')
        chg24 = md.get('price_change_24h')
        if price:
            print(f'  Preis: ${price:,.2f}', end='')
        if chg24 is not None:
            sign = '+' if chg24 >= 0 else ''
            print(f'  ({sign}{chg24:.2f}% 24h)', end='')
        if cap:
            print(f'  | Market Cap: ${cap/1e9:.1f}B')
        print()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ZELLE 6: Charts anzeigen (direkte Analyse)
# ═══════════════════════════════════════════════════════════════════════
# Zeigt die Analyse-Charts direkt im Notebook.
# Führe zuerst Zelle 5 aus!

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd

if 'result' not in dir() or result.ohlcv.empty:
    print('❌ Führe zuerst Zelle 5 aus!')
else:
    df = result.ohlcv

    # ── Candlestick Chart ─────────────────────────────────────────────
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.6, 0.2, 0.2],
        subplot_titles=[f'{SYMBOL} Kurs', 'RSI(14)', 'Volumen'],
    )

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['open'], high=df['high'],
        low=df['low'], close=df['close'],
        name='Kurs',
    ), row=1, col=1)

    # EMA
    for ema_col, color, name in [
        ('ema_21', '#2196F3', 'EMA21'),
        ('ema_50', '#FF9800', 'EMA50'),
    ]:
        if ema_col in df.columns:
            fig.add_trace(go.Scatter(
                x=df.index, y=df[ema_col],
                name=name, line=dict(color=color, width=1.5),
            ), row=1, col=1)

    # Bollinger Bands
    for bb_col in ['bb_upper', 'bb_lower']:
        if bb_col in df.columns:
            fig.add_trace(go.Scatter(
                x=df.index, y=df[bb_col],
                name=bb_col.replace('_', ' ').title(),
                line=dict(color='rgba(128,128,128,0.4)', width=1, dash='dot'),
            ), row=1, col=1)

    # RSI
    if 'rsi_14' in df.columns:
        fig.add_trace(go.Scatter(
            x=df.index, y=df['rsi_14'],
            name='RSI(14)', line=dict(color='#9C27B0', width=1.5),
        ), row=2, col=1)
        fig.add_hline(y=70, line_dash='dash', line_color='red', opacity=0.5, row=2, col=1)
        fig.add_hline(y=30, line_dash='dash', line_color='green', opacity=0.5, row=2, col=1)

    # Volumen
    colors = ['#1a7f37' if c >= o else '#c82538'
              for c, o in zip(df['close'], df['open'])]
    fig.add_trace(go.Bar(
        x=df.index, y=df['volume'],
        name='Volumen', marker_color=colors,
    ), row=3, col=1)

    fig.update_layout(
        height=700,
        title=f'{SYMBOL} – Technische Analyse',
        xaxis_rangeslider_visible=False,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(15,17,23,0.9)',
    )
    fig.show()

    # ── Backtest Chart ────────────────────────────────────────────────
    bt = result.backtest
    if bt is not None:
        print(f'\n📈 Backtest-Ergebnis ({bt.period_label}):')
        print(f'   Strategie:  {bt.total_return_pct:+.1f}%')
        print(f'   Buy & Hold: {bt.bnh_return_pct:+.1f}%')
        print(f'   Alpha:      {bt.alpha_pct:+.1f}pp')
        print(f'   Max DD:     {bt.max_drawdown_pct:.1f}%')
        print(f'   Sharpe:     {bt.sharpe_ratio:.2f}')

        fig_bt = go.Figure()
        fig_bt.add_trace(go.Scatter(
            x=bt.series.index, y=bt.series['portfolio'],
            name='KI-Strategie', line=dict(color='#2196F3', width=2.5),
        ))
        fig_bt.add_trace(go.Scatter(
            x=bt.series.index, y=bt.series['benchmark'],
            name='Buy & Hold', line=dict(color='#FF9800', width=2, dash='dash'),
        ))
        fig_bt.add_hline(y=100, line_dash='dot', line_color='gray', opacity=0.5)
        fig_bt.update_layout(
            title=f'{SYMBOL} Backtest – Strategie vs. Buy & Hold',
            height=350,
            yaxis_title='Portfolio-Wert (Start = 100)',
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(15,17,23,0.9)',
        )
        fig_bt.show()
    else:
        print('\nKein Backtest verfügbar (zu wenig Walk-Forward-Daten)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ZELLE 7: Mehrere Coins scannen (Screener)
# ═══════════════════════════════════════════════════════════════════════
# Scannt mehrere Coins mit technischen Indikatoren und zeigt eine
# Übersichtstabelle. Führe zuerst Zellen 1+2 aus!

import sys, os
PROJECT_DIR = '/content/crypto_analyzer'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

from pathlib import Path
from src.analysis.analyzer import CryptoAnalyzer
from src.features.technical import TechnicalIndicators
import yaml, pandas as pd

# ─── Symbole zum Scannen ──────────────────────────────────────────────
SCAN_SYMBOLS = ['BTC', 'ETH', 'SOL', 'BNB', 'XRP', 'ADA', 'DOGE', 'AVAX']
SCAN_INTERVAL = '1d'
SCAN_LOOKBACK = 180
# ─────────────────────────────────────────────────────────────────────

config_path = Path(PROJECT_DIR) / 'config.yaml'
with config_path.open() as f:
    config = yaml.safe_load(f)

analyzer = CryptoAnalyzer(config_path)
ti = TechnicalIndicators(config=config)

rows = []
print(f'🔍 Scanne {len(SCAN_SYMBOLS)} Coins...\n')

for i, sym in enumerate(SCAN_SYMBOLS, 1):
    print(f'  [{i}/{len(SCAN_SYMBOLS)}] {sym}', end=' ')
    try:
        df = analyzer._fetcher.get_ohlcv(sym, SCAN_INTERVAL, SCAN_LOOKBACK)
        if df.empty or len(df) < 30:
            print('❌ (zu wenig Daten)')
            continue

        df = ti.add_all(df)
        last = df.iloc[-1]

        rsi = last.get('rsi_14', 50)
        macd_hist = last.get('macd_diff', 0)
        bb_pct = last.get('bb_pct', 0.5)
        ema9 = last.get('ema_9')
        ema21 = last.get('ema_21')

        bull = (int(rsi < 40) + int(macd_hist > 0) +
                int(ema9 is not None and ema21 is not None and ema9 > ema21) +
                int(bb_pct < 0.25))
        bear = (int(rsi > 65) + int(macd_hist < 0) +
                int(ema9 is not None and ema21 is not None and ema9 < ema21) +
                int(bb_pct > 0.80))

        signal = 'BULLISH' if bull >= 3 else ('BEARISH' if bear >= 3 else 'NEUTRAL')
        emoji = {'BULLISH': '🟢', 'BEARISH': '🔴', 'NEUTRAL': '🟡'}[signal]

        price_7d = None
        if len(df) >= 8:
            price_7d = (df['close'].iloc[-1] / df['close'].iloc[-8] - 1) * 100

        rows.append({
            'Symbol': sym,
            'Preis': f"${last['close']:,.4f}",
            '7T Änderung': f"{price_7d:+.1f}%" if price_7d is not None else '–',
            'RSI(14)': round(float(rsi), 1),
            'Signal': f"{emoji} {signal}",
            'ADX': round(float(last.get('adx', 0)), 1),
        })
        print(f'→ {emoji} {signal}')

    except Exception as e:
        print(f'❌ ({e})')

if rows:
    print('\n' + '═'*60)
    print('  SCREENER-ERGEBNIS')
    print('═'*60)
    scan_df = pd.DataFrame(rows)
    # Farbige Anzeige
    from IPython.display import display, HTML

    def color_signal(val):
        if 'BULLISH' in str(val):
            return 'background-color: rgba(26,127,55,0.2); color: #1a7f37; font-weight: bold'
        elif 'BEARISH' in str(val):
            return 'background-color: rgba(200,37,56,0.2); color: #c82538; font-weight: bold'
        return 'color: #b08800'

    def color_change(val):
        try:
            if float(str(val).replace('%','').replace('+','')) > 0:
                return 'color: #1a7f37'
            return 'color: #c82538'
        except Exception:
            return ''

    styled = scan_df.style.applymap(color_signal, subset=['Signal']).applymap(color_change, subset=['7T Änderung'])
    display(styled)

    # Zusammenfassung
    bullish = [r['Symbol'] for r in rows if 'BULLISH' in r['Signal']]
    bearish = [r['Symbol'] for r in rows if 'BEARISH' in r['Signal']]
    print(f'\n🟢 Bullische Coins: {', '.join(bullish) if bullish else 'keine'}')
    print(f'🔴 Bärische Coins: {', '.join(bearish) if bearish else 'keine'}')
    print('\n⚠️  Technische Signale sind kein Anlageauftrag!')
else:
    print('\n❌ Keine Daten verfügbar. Internetverbindung prüfen.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ZELLE 8: App stoppen (Aufräumen)
# ═══════════════════════════════════════════════════════════════════════
# Führe diese Zelle aus, wenn du die App beenden möchtest.

try:
    from pyngrok import ngrok
    ngrok.kill()
    print('✅ ngrok-Tunnel geschlossen')
except Exception as e:
    print(f'ℹ️  ngrok bereits inaktiv: {e}')

try:
    streamlit_proc.terminate()
    print('✅ Streamlit-Prozess beendet')
except Exception as e:
    print(f'ℹ️  Streamlit bereits inaktiv: {e}')

print('\n👋 App gestoppt. Auf Wiedersehen!')

---
## Häufige Probleme & Lösungen

| Problem | Lösung |
|---|---|
| URL öffnet sich nicht | Warte 10s und lade die Seite neu |
| ngrok-Fehler | Trage einen kostenlosen Token in Zelle 3 ein |
| `ModuleNotFoundError` | Führe Zelle 1 erneut aus, dann Runtime neu starten |
| App ist sehr langsam | Erster Coin-Scan trainiert das KI-Modell (~60s). Danach wird gecacht. |
| Kein Signal angezeigt | Erhöhe 'Historische Tage' auf 500+ in der App |
| Session abgelaufen | Alle Zellen erneut ausführen |

## Hilfreiche Links

- 🐙 [GitHub Repository](https://github.com/bademeischta/crypto_analyzer)
- 🔑 [Kostenloser ngrok-Account](https://dashboard.ngrok.com/signup)
- 📖 [Streamlit Docs](https://docs.streamlit.io)

---
⚠️ **Rechtlicher Hinweis:** Diese Anwendung dient ausschließlich Bildungszwecken und stellt
keine Finanzberatung dar. Kryptowährungen sind hochspekulative Anlagen. Vergangene
Performance garantiert keine zukünftigen Ergebnisse. Alle Entscheidungen liegen
beim Nutzer.